# Agentic AI — Gestor Personal de Actividades


| Herramienta | Descripcion |
|---|---|
| 📝 Tareas | Crear, listar, completar y eliminar |
| 📅 Agenda | Agendar y consultar eventos |
| ⏰ Recordatorios | Crear recordatorios con fecha |
| 📖 Diario | Notas y reflexiones personales |
| 🔍 Busqueda | Buscar en todas las actividades |
| 📊 Resumen | Reporte de productividad |


## ⚙️ Instalacion de dependencias

In [79]:
!pip install -q google-generativeai>=0.8.0 python-dateutil ipywidgets

## 🔑 Configurar API Key de Gemini

> Obtener clave en: https://aistudio.google.com/app/apikey

> **En Colab:** Panel izquierdo → 🔑 Secretos → Agrega `GEMINI_API_KEY`

In [91]:
import os
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ API Key cargada desde Colab Secrets')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
if not GEMINI_API_KEY:
    GEMINI_API_KEY = input('🔑 Ingresa tu Gemini API Key: ').strip()
print('✅ API Key lista') if GEMINI_API_KEY else print('❌ Sin API Key')

✅ API Key cargada desde Colab Secrets
✅ API Key lista


## 🛠️ Herramientas del Agente

En este paso se definen las funciones que el agente puede "llamar" para realizar acciones específicas, como crear una tarea, agendar un evento, o escribir en el diario. Estas funciones interactúan con una base de datos interna (`DB`). Cada función está diseñada para realizar una acción concreta y devolver un resultado estructurado.

In [92]:
import json, uuid
from datetime import datetime
from dateutil import parser as dateparser

DB = {'tasks': [], 'events': [], 'reminders': [], 'diary': []}

def _now():
    return datetime.now().strftime('%Y-%m-%d %H:%M')

def _parse_date(s):
    try:
        return dateparser.parse(s, dayfirst=True).strftime('%Y-%m-%d %H:%M')
    except Exception:
        return s

# === TAREAS ====
def crear_tarea(titulo, descripcion='', prioridad='Media', fecha_limite='', etiquetas=None):
    t = {'id': str(uuid.uuid4())[:8], 'titulo': titulo, 'descripcion': descripcion,
         'prioridad': prioridad, 'completada': False, 'fecha_creacion': _now(),
         'fecha_limite': _parse_date(fecha_limite) if fecha_limite else '',
         'etiquetas': etiquetas or []}
    DB['tasks'].append(t)
    return {'success': True, 'mensaje': f"Tarea '{titulo}' creada (ID: {t['id']})", 'tarea': t}

def listar_tareas(filtro_estado='todas', filtro_prioridad=''):
    ts = list(DB['tasks'])
    if filtro_estado == 'pendientes':    ts = [t for t in ts if not t['completada']]
    elif filtro_estado == 'completadas': ts = [t for t in ts if t['completada']]
    if filtro_prioridad: ts = [t for t in ts if t['prioridad'].lower() == filtro_prioridad.lower()]
    return {'success': True, 'total': len(ts), 'tareas': ts}

def completar_tarea(task_id):
    for t in DB['tasks']:
        if t['id'] == task_id:
            t.update({'completada': True, 'fecha_completada': _now()})
            return {'success': True, 'mensaje': f"Tarea '{t['titulo']}' completada"}
    return {'success': False, 'mensaje': f'Tarea {task_id} no encontrada'}

def eliminar_tarea(task_id):
    antes = len(DB['tasks'])
    DB['tasks'] = [t for t in DB['tasks'] if t['id'] != task_id]
    ok = len(DB['tasks']) < antes
    return {'success': ok, 'mensaje': 'Tarea eliminada' if ok else 'No encontrada'}

# === AGENDA ===
def agendar_evento(titulo, fecha_inicio, fecha_fin='', descripcion='', lugar='', participantes=None):
    e = {'id': str(uuid.uuid4())[:8], 'titulo': titulo, 'descripcion': descripcion,
         'fecha_inicio': _parse_date(fecha_inicio),
         'fecha_fin': _parse_date(fecha_fin) if fecha_fin else '',
         'lugar': lugar, 'participantes': participantes or [], 'creado_en': _now()}
    DB['events'].append(e)
    return {'success': True, 'mensaje': f"Evento '{titulo}' agendado para {e['fecha_inicio']}", 'evento': e}

def consultar_agenda(fecha='', dias_adelante=7):
    evs = sorted(DB['events'], key=lambda x: x['fecha_inicio'])
    return {'success': True, 'total': len(evs), 'eventos': evs}

def cancelar_evento(event_id):
    antes = len(DB['events'])
    DB['events'] = [e for e in DB['events'] if e['id'] != event_id]
    ok = len(DB['events']) < antes
    return {'success': ok, 'mensaje': 'Evento cancelado' if ok else 'No encontrado'}

# === RECORDATORIOS ====
def crear_recordatorio(mensaje, fecha_hora, repetir='no'):
    r = {'id': str(uuid.uuid4())[:8], 'mensaje': mensaje,
         'fecha_hora': _parse_date(fecha_hora), 'repetir': repetir,
         'activo': True, 'creado_en': _now()}
    DB['reminders'].append(r)
    return {'success': True, 'mensaje': f"Recordatorio para {r['fecha_hora']}", 'recordatorio': r}

def listar_recordatorios():
    activos = [r for r in DB['reminders'] if r['activo']]
    return {'success': True, 'total': len(activos), 'recordatorios': activos}

def eliminar_recordatorio(reminder_id):
    for r in DB['reminders']:
        if r['id'] == reminder_id:
            r['activo'] = False
            return {'success': True, 'mensaje': 'Recordatorio eliminado'}
    return {'success': False, 'mensaje': 'No encontrado'}

# ==== DIARIO ====
def escribir_en_diario(contenido, estado_animo='neutral', etiquetas=None):
    d = {'id': str(uuid.uuid4())[:8], 'contenido': contenido,
         'estado_animo': estado_animo, 'etiquetas': etiquetas or [], 'fecha': _now()}
    DB['diary'].append(d)
    return {'success': True, 'mensaje': 'Entrada guardada', 'entrada': d}

def leer_diario(cantidad=5):
    return {'success': True, 'entradas': DB['diary'][-cantidad:]}

# ==== BUSQUEDA Y RESUMEN =====
def buscar(query):
    q = query.lower()
    return {'success': True, 'query': query, 'resultados': {
        'tareas':        [t for t in DB['tasks']     if q in t['titulo'].lower() or q in t['descripcion'].lower()],
        'eventos':       [e for e in DB['events']    if q in e['titulo'].lower() or q in e['descripcion'].lower()],
        'recordatorios': [r for r in DB['reminders'] if q in r['mensaje'].lower()],
        'diario':        [d for d in DB['diary']     if q in d['contenido'].lower()],
    }}

def generar_resumen():
    pend  = [t for t in DB['tasks'] if not t['completada']]
    comp  = [t for t in DB['tasks'] if t['completada']]
    total = len(DB['tasks'])
    return {'success': True, 'resumen': {
        'tareas': {'total': total, 'pendientes': len(pend), 'completadas': len(comp),
                   'alta_prioridad': len([t for t in pend if t['prioridad']=='Alta']),
                   'tasa': f'{len(comp)/total*100:.0f}%' if total else '0%'},
        'eventos': {'total': len(DB['events'])},
        'recordatorios': {'activos': len([r for r in DB['reminders'] if r['activo']])},
        'diario': {'entradas': len(DB['diary'])}
    }}

def obtener_fecha_hora_actual():
    n = datetime.now()
    return {'success': True, 'fecha': n.strftime('%Y-%m-%d'),
            'hora': n.strftime('%H:%M'), 'dia_semana': n.strftime('%A'), 'timestamp': n.isoformat()}

FUNCTION_MAP = {
    'crear_tarea': crear_tarea, 'listar_tareas': listar_tareas,
    'completar_tarea': completar_tarea, 'eliminar_tarea': eliminar_tarea,
    'agendar_evento': agendar_evento, 'consultar_agenda': consultar_agenda, 'cancelar_evento': cancelar_evento,
    'crear_recordatorio': crear_recordatorio, 'listar_recordatorios': listar_recordatorios,
    'eliminar_recordatorio': eliminar_recordatorio,
    'escribir_en_diario': escribir_en_diario, 'leer_diario': leer_diario,
    'buscar': buscar, 'generar_resumen': generar_resumen,
    'obtener_fecha_hora_actual': obtener_fecha_hora_actual,
}

## 🤖 Configurar Gemini con Function Calling

Se le proporcionan las `FunctionDeclaration` que describen cada herramienta (nombre, descripción, parámetros). También se define un `SYSTEM_PROMPT` que guía el comportamiento general del agente, asegurando que responda en español, use emojis, sea amigable y utilice las herramientas de forma proactiva.

In [95]:
import google.generativeai as genai
from google.generativeai.types import FunctionDeclaration, Tool

genai.configure(api_key=GEMINI_API_KEY)

tools_def = [
    FunctionDeclaration(name='crear_tarea', description='Crea una nueva tarea personal',
        parameters={'type':'object','properties':{
            'titulo':{'type':'string'},'descripcion':{'type':'string'},
            'prioridad':{'type':'string','enum':['Alta','Media','Baja']},
            'fecha_limite':{'type':'string'},'etiquetas':{'type':'array','items':{'type':'string'}}
        },'required':['titulo']}),
    FunctionDeclaration(name='listar_tareas', description='Lista tareas con filtros',
        parameters={'type':'object','properties':{
            'filtro_estado':{'type':'string','enum':['todas','pendientes','completadas']},
            'filtro_prioridad':{'type':'string'}}}),
    FunctionDeclaration(name='completar_tarea', description='Marca tarea como completada',
        parameters={'type':'object','properties':{'task_id':{'type':'string'}},'required':['task_id']}),
    FunctionDeclaration(name='eliminar_tarea', description='Elimina una tarea',
        parameters={'type':'object','properties':{'task_id':{'type':'string'}},'required':['task_id']}),
    FunctionDeclaration(name='agendar_evento', description='Agenda un evento en el calendario',
        parameters={'type':'object','properties':{
            'titulo':{'type':'string'},'fecha_inicio':{'type':'string'},
            'fecha_fin':{'type':'string'},'descripcion':{'type':'string'},
            'lugar':{'type':'string'},'participantes':{'type':'array','items':{'type':'string'}}
        },'required':['titulo','fecha_inicio']}),
    FunctionDeclaration(name='consultar_agenda', description='Consulta eventos proximos',
        parameters={'type':'object','properties':{'fecha':{'type':'string'},'dias_adelante':{'type':'integer'}}}),
    FunctionDeclaration(name='cancelar_evento', description='Cancela un evento',
        parameters={'type':'object','properties':{'event_id':{'type':'string'}},'required':['event_id']}),
    FunctionDeclaration(name='crear_recordatorio', description='Crea un recordatorio',
        parameters={'type':'object','properties':{
            'mensaje':{'type':'string'},'fecha_hora':{'type':'string'},
            'repetir':{'type':'string','enum':['no','diario','semanal','mensual']}
        },'required':['mensaje','fecha_hora']}),
    FunctionDeclaration(name='listar_recordatorios', description='Lista recordatorios activos',
        parameters={'type':'object','properties':{}}),
    FunctionDeclaration(name='eliminar_recordatorio', description='Elimina un recordatorio',
        parameters={'type':'object','properties':{'reminder_id':{'type':'string'}},'required':['reminder_id']}),
    FunctionDeclaration(name='escribir_en_diario', description='Guarda entrada en el diario',
        parameters={'type':'object','properties':{
            'contenido':{'type':'string'},'estado_animo':{'type':'string'},
            'etiquetas':{'type':'array','items':{'type':'string'}}
        },'required':['contenido']}),
    FunctionDeclaration(name='leer_diario', description='Lee entradas del diario',
        parameters={'type':'object','properties':{'cantidad':{'type':'integer'}}}),
    FunctionDeclaration(name='buscar', description='Busca en todas las actividades',
        parameters={'type':'object','properties':{'query':{'type':'string'}},'required':['query']}),
    FunctionDeclaration(name='generar_resumen', description='Resumen de productividad',
        parameters={'type':'object','properties':{}}),
    FunctionDeclaration(name='obtener_fecha_hora_actual', description='Fecha y hora actual',
        parameters={'type':'object','properties':{}}),
]

SYSTEM_PROMPT = (
    'Eres un asistente personal inteligente para gestion de productividad. '
    'Ayudas con tareas, agenda, recordatorios y diario personal. '
    'SIEMPRE usa las herramientas para acciones concretas. '
    'Responde SIEMPRE en espanol, con emojis y tono amigable. '
    'Para fechas relativas (hoy, mañana), usa obtener_fecha_hora_actual primero. '
    'Se proactivo: si dicen reunion manana, crea el evento Y sugiere preparacion.'
)

model = genai.GenerativeModel(
    model_name='gemini-2.5-flash',
    tools=[Tool(function_declarations=tools_def)],
    system_instruction=SYSTEM_PROMPT
)

print(f'Gemini ({model.model_name}) configurado con {len(tools_def)} herramientas')

Gemini (models/gemini-2.5-flash) configurado con 15 herramientas


## 🧠 Motor Agentico ReAct

Se implementa la lógica del agente, en base a el patrón **ReAct (Reasoning and Acting)

1. **Recibir un mensaje** del usuario.
2. **Razonar** sobre qué herramientas necesita usar para responder al mensaje.
3. **Actuar** llamando a las herramientas seleccionadas.
4. **Procesar los resultados** de las herramientas.
5. **Generar una respuesta** final al usuario.

El ciclo se repite hasta que el agente considera que ha resuelto la consulta o ha alcanzado el límite de iteraciones.

In [97]:
from IPython.display import display, Markdown, HTML

historial_chat = []

ICONOS = {
    'crear_tarea':'📝','listar_tareas':'📋','completar_tarea':'✅','eliminar_tarea':'🗑️',
    'agendar_evento':'📅','consultar_agenda':'🗓️','cancelar_evento':'❌',
    'crear_recordatorio':'⏰','listar_recordatorios':'⏰','eliminar_recordatorio':'🗑️',
    'escribir_en_diario':'📖','leer_diario':'📖','buscar':'🔍',
    'generar_resumen':'📊','obtener_fecha_hora_actual':'🕐',
}

def _convertir_args(args):
    """Convierte objetos protobuf de Gemini a tipos Python nativos."""
    resultado = {}
    for k, v in args.items():
        if hasattr(v, '__class__') and 'Repeated' in type(v).__name__:
            resultado[k] = list(v)
        elif isinstance(v, dict):
            resultado[k] = _convertir_args(v)
        else:
            resultado[k] = v
    return resultado

def _ejecutar_fn(nombre, args):
    if nombre not in FUNCTION_MAP:
        return {'success': False, 'error': f'Funcion {nombre} no existe'}
    try:
        args_limpios = _convertir_args(dict(args))
        return FUNCTION_MAP[nombre](**args_limpios)
    except Exception as ex:
        return {'success': False, 'error': str(ex)}

def _mostrar_tool(nombre):
    ico = ICONOS.get(nombre, '🔧')
    display(HTML(
        '<div style="background:#1a1a2e;color:#00d4aa;padding:6px 12px;'
        'border-radius:6px;font-family:monospace;font-size:12px;'
        'margin:3px 0;border-left:3px solid #00d4aa">'
        f'{ico} <b>Tool:</b> {nombre}()</div>'
    ))

def agente(mensaje, historial=None, max_iter=10):
    global historial_chat
    if historial is None:
        historial = historial_chat
    chat = model.start_chat(history=historial)
    resp = chat.send_message(mensaje)
    for _ in range(max_iter):
        calls = [p for p in resp.parts if hasattr(p, 'function_call') and p.function_call.name]
        if not calls:
            break
        tool_parts = []
        for p in calls:
            fc = p.function_call
            _mostrar_tool(fc.name)
            resultado = _ejecutar_fn(fc.name, dict(fc.args))
            tool_parts.append(genai.protos.Part(
                function_response=genai.protos.FunctionResponse(
                    name=fc.name,
                    response={'result': json.dumps(resultado, ensure_ascii=False)}
                )))
        resp = chat.send_message(tool_parts)
    texto = ''.join(p.text for p in resp.parts if hasattr(p, 'text') and p.text)
    historial_chat = list(chat.history)
    return texto

def chat_con_agente(msg):
    display(HTML(
        '<div style="text-align:right;margin:8px 0">'
        f'<span style="background:#4299e1;color:white;padding:8px 14px;'
        'border-radius:18px 18px 4px 18px;font-size:14px">'
        f'{msg}</span> 👤</div>'
    ))
    display(HTML('<div style="color:#718096;font-size:13px">🧠 Procesando...</div>'))
    try:
        respuesta = agente(msg)
        display(Markdown('🤖 ' + respuesta))
    except Exception as ex:
        display(HTML(f'<div style="color:red">❌ Error: {ex}</div>'))
    display(HTML('<hr style="border-color:#e2e8f0;margin:10px 0">'))

## 💬 Chat Interactivo

Crea una interfaz de usuario interactiva utilizando `ipywidgets`.

Incluye un área de salida para mostrar la conversación, un campo de texto para escribir mensajes y botones de envío y limpieza, así como botones de acceso rápido para preguntas comunes.

In [102]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

output_area = widgets.Output(layout=widgets.Layout(
    border='1px solid #e2e8f0', min_height='380px', max_height='540px',
    overflow_y='auto', padding='14px', border_radius='10px'))

txt = widgets.Text(
    placeholder='Escribe tu mensaje... ej: Crea tarea urgente revisar emails',
    layout=widgets.Layout(width='78%', height='38px'))

btn_send  = widgets.Button(description='Enviar 🚀', button_style='primary',
    layout=widgets.Layout(width='12%', height='38px'))
btn_clear = widgets.Button(description='🗑️ Limpiar', button_style='warning',
    layout=widgets.Layout(width='10%', height='38px'))

QUICK = [
    ('📊 Resumen',       'Dame un resumen completo de mi productividad'),
    ('📋 Mis Tareas',    'Lista todas mis tareas pendientes por prioridad'),
    ('🗓️ Mi Agenda',    'Muestra mis eventos programados'),
    ('⏰ Recordatorios', 'Lista mis recordatorios activos'),
    ('📖 Diario',        'Lee mis ultimas entradas del diario'),
]

quick_btns = [widgets.Button(description=lbl, button_style='info',
    layout=widgets.Layout(margin='2px')) for lbl, _ in QUICK]

def on_send(b=None):
    msg = txt.value.strip()
    if not msg: return
    txt.value = ''
    btn_send.disabled = True
    btn_send.description = '⏳...'
    with output_area:
        chat_con_agente(msg)
    btn_send.disabled = False
    btn_send.description = 'Enviar 🚀'

def on_clear(b=None):
    global historial_chat
    historial_chat = []
    output_area.clear_output()
    with output_area:
        display(HTML(
            '<div style="background:linear-gradient(135deg,#0f0c29,#302b63);'
            'padding:20px;border-radius:12px;color:white;font-family:sans-serif">'
            '<h3 style="margin:0">🤖 Agentic AI Chat</h3>'
            '<p style="color:#a0aec0;margin:6px 0 0">Habla en lenguaje natural</p>'
            '</div>'
        ))

def make_quick(idx):
    def h(b):
        txt.value = QUICK[idx][1]; on_send()
    return h

btn_send.on_click(on_send)
btn_clear.on_click(on_clear)
txt.on_submit(lambda x: on_send())
for i, b in enumerate(quick_btns):
    b.on_click(make_quick(i))

header = widgets.HTML(
    '<div style="background:linear-gradient(135deg,#0f0c29,#302b63);'
    'padding:16px;border-radius:12px;color:white;margin-bottom:10px">'
    '<h3 style="margin:0">🤖 Asistente Agentic AI — Usando Gemini</h3>'
    '<p style="color:#a0aec0;font-size:12px;margin:4px 0 0">'
    'Creditos limitados por tiempo</p>'
    '</div>'
)

ui = widgets.VBox([
    header,
    widgets.HTML('<small style="color:#718096">⚡ Acciones rapidas:</small>'),
    widgets.HBox(quick_btns, layout=widgets.Layout(flex_wrap='wrap', margin='4px 0')),
    output_area,
    widgets.HBox([txt, btn_send, btn_clear])
])

on_clear()
display(ui)

### Ejemplos de prueba

In [103]:
# Descomenta cualquier linea para probar directamente:
# chat_con_agente('Crea tres tareas: revisar emails (alta), preparar presentacion (media), llamar medico (baja)')
# chat_con_agente('Agendame reunion de equipo manana a las 10am en sala de conferencias con Juan y Maria')
# chat_con_agente('Recordatorio: tomar vitaminas todos los dias a las 8am')
# chat_con_agente('Escribe en mi diario: hoy fue un dia muy productivo, me siento motivado')
# chat_con_agente('Dame mi resumen de productividad')
# chat_con_agente('Busca todo sobre la reunion')

## 💾 Guardar y Cargar Datos (En construcción)

> Guarda tus actividades en JSON para no perderlas entre sesiones

In [105]:
from pathlib import Path
from IPython.display import display, Markdown

DB_FILE = 'gestor_personal.json'

def guardar_datos():
    with open(DB_FILE, 'w', encoding='utf-8') as f:
        json.dump(DB, f, ensure_ascii=False, indent=2)
    print(f'✅ Datos guardados en {DB_FILE}')
    print(f'   📝 Tareas: {len(DB["tasks"])}, 📅 Eventos: {len(DB["events"])}')
    print(f'   ⏰ Recordatorios: {len(DB["reminders"])}, 📖 Diario: {len(DB["diary"])}')

def cargar_datos():
    global DB
    if Path(DB_FILE).exists():
        with open(DB_FILE, 'r', encoding='utf-8') as f:
            DB = json.load(f)
        print(f'✅ Datos cargados desde {DB_FILE}')
        print(f'   📝 Tareas: {len(DB["tasks"])}, 📅 Eventos: {len(DB["events"])}')
    else:
        print(f'⚠️ {DB_FILE} no encontrado. Base de datos vacia.')

def exportar_resumen_md():
    n = datetime.now().strftime('%Y-%m-%d %H:%M')
    lineas = [f'# Resumen Personal {n}', '', '## Tareas', '']
    for t in DB['tasks']:
        ok = '✅' if t['completada'] else '⬜'
        lineas.append(f'- {ok} [{t["prioridad"]}] {t["titulo"]}')
    lineas += ['', '## Eventos', '']
    for e in DB['events']:
        lineas.append(f'- 📅 {e["titulo"]} — {e["fecha_inicio"]}')
    lineas += ['', '## Recordatorios', '']
    for r in DB['reminders']:
        if r['activo']:
            lineas.append(f'- ⏰ {r["mensaje"]} — {r["fecha_hora"]}')
    md = chr(10).join(lineas)
    with open('resumen_personal.md', 'w', encoding='utf-8') as f:
        f.write(md)
    print('✅ Exportado a resumen_personal.md')
    display(Markdown(md))

cargar_datos()

⚠️ gestor_personal.json no encontrado. Base de datos vacia.


---

## Comandos de referencia

```python
guardar_datos()        # Guarda tus datos
cargar_datos()         # Carga datos previos
exportar_resumen_md()  # Exporta reporte
chat_con_agente('...')  # Chat directo
```

### IA Agentica vs. Chatbot

-   **Chatbot:** Conversa, responde preguntas basándose en su conocimiento preentrenado o reglas. Es **reactivo** y no puede realizar acciones externas.
-   **IA Agentica (con patrón ReAct):** No solo conversa, sino que **razona** para entender el objetivo, **actúa** ejecutando herramientas (crear tareas, agendar eventos, etc.), y **observa** los resultados para adaptar su plan. Es **proactivo** y orientado a objetivos, interactuando con el "mundo" para cumplir la solicitud del usuario.